# DS-05: Re-entrenar con datos nuevos (copia de 02_vectorizacion_modelo.ipynb)

**TAREA TRELLO: SPRINT-S1_DS-05: Re-entrenar con datos nuevos**

**Proyecto:** TechKnowledge API -- Hackathon ONE Alura Latam x Oracle
**Objetivo:** Vectorizar el corpus de contenidos tecnicos con TF-IDF (incluyendo bigramas), entrenar un clasificador
de Regresion Logistica, evaluar su desempeno con metricas estandar y serializar el modelo final para el equipo de Backend.

Esta es una **copia** de `02_vectorizacion_modelo.ipynb` (el original queda intacto), pensada para re-entrenar
con el dataset ampliado/actualizado sin perder la version que genero el modelo actualmente en produccion. Misma
metodologia exacta (Config B: TF-IDF 10000 features/min_df=2 + oversampling + Logistic Regression), la unica
diferencia de configuracion es que las stopwords ahora se importan desde `shared/stop_words.py` del repo en vez
de la lista hardcodeada anterior (mejora validada de accuracy y confianza en ambos idiomas).

**Como correr este notebook para los 2 idiomas:** el notebook entrena **un modelo por idioma, en corridas separadas**.
Corre todo el notebook (`Run All`) con `IDIOMA = "en"` -> obtienes el modelo en ingles serializado en `../model/en/`.
Despues cambia `IDIOMA = "es"` y corre todo el notebook de nuevo (`Run All`) -> obtienes el modelo en espanol en
`../model/es/`. No se cargan ni se mezclan los dos idiomas en la misma corrida.

Este notebook espera como entrada el `dataset_limpio_<idioma>.csv` que genera `Limpieza_Datos_NLP.ipynb` (correlo
primero, con el dataset nuevo, para el mismo idioma).

**Contenido de este notebook:**
- [x] Seleccionar idioma (EN/ES) y cargar el dataset limpio correspondiente
- [x] Construir el corpus (`titulo` + `texto`)
- [x] Train/test split estratificado
- [x] Balanceo de clases por oversampling (solo en el set de entrenamiento, para evitar fuga de datos)
- [x] Vectorizacion TF-IDF con bigramas (stopwords en ingles o espanol, importadas desde shared/stop_words.py)
- [x] Entrenamiento de Regresion Logistica
- [x] Evaluacion: accuracy, precision, recall, F1-score, matriz de confusion
- [x] Serializacion del modelo (`joblib`), separada por idioma


## 0. Configuracion para Google Colab

Si estas corriendo este notebook en **Google Colab**, ejecuta esta celda para subir el dataset directamente
(omitela si lo estas corriendo localmente con el archivo ya en `../data/`).

In [ ]:
# Descomenta estas lineas si estas en Google Colab y necesitas subir el CSV manualmente
# (sube el dataset_limpio_<idioma>.csv que corresponda al IDIOMA que vas a procesar en esta corrida)
# from google.colab import files
# uploaded = files.upload()


## 1. Importar librerias

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

pd.set_option('display.max_colwidth', 100)


### 1.1 Seleccion de idioma

Este notebook entrena un modelo a la vez. Cambia `IDIOMA` a `"en"` o `"es"` y ejecuta todo el notebook (`Run All`)
para entrenar y serializar el modelo de ese idioma. Para tener los dos modelos (EN y ES), se corre el notebook dos
veces, una por idioma -- son entrenamientos independientes, no se combinan los datos de los dos idiomas.

In [ ]:
# Selecciona el idioma a entrenar: "en" (ingles) o "es" (espanol)
IDIOMA = "en"  # cambiar a "es" para entrenar el modelo en espanol

archivos_entrada = {
    "en": "dataset_limpio_en.csv",
    "es": "dataset_limpio_es.csv",
}
assert IDIOMA in archivos_entrada, "IDIOMA debe ser 'en' o 'es'"

# Stopwords: se importan desde el modulo compartido del repo (shared/stop_words.py),
# curado por el equipo (~750 palabras EN / ~950 ES) -- una sola fuente de verdad,
# usada tanto por los notebooks de Colab como por la API de FastAPI.
# Se valido que mejora accuracy y confianza de forma consistente en EN y ES
# respecto a la lista anterior (SPANISH_STOPWORDS hardcodeada / 'english' nativa).
!curl -sL -o shared_stop_words.py "https://raw.githubusercontent.com/No-Country-simulation/g9-latam-team-04-esp/develop/shared/stop_words.py"
from shared_stop_words import STOP_WORDS_EN, STOP_WORDS_ES

stop_words = STOP_WORDS_EN if IDIOMA == "en" else STOP_WORDS_ES

print(f"Idioma seleccionado: {IDIOMA.upper()}")
print(f"Archivo de entrada: {archivos_entrada[IDIOMA]}")
print(f"Stopwords cargadas: {len(stop_words)} palabras ({'EN' if IDIOMA == 'en' else 'ES'})")


## 2. Cargar el dataset limpio

Cargamos `dataset_limpio.csv`. Este archivo ya contiene, para cada contenido: titulo, texto, categoria,
ademas de metadatos adicionales (keywords, longitud, url) generados en pasos previos del equipo.

In [ ]:
nombre_archivo = archivos_entrada[IDIOMA]

try:
    file_path = f"../data/{nombre_archivo}"
    df = pd.read_csv(file_path)
except FileNotFoundError:
    file_path = f"/content/{nombre_archivo}"
    df = pd.read_csv(file_path)

print(f"Idioma: {IDIOMA.upper()}")
print(f"Dataset cargado desde: {file_path}")
print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"\nColumnas disponibles: {df.columns.tolist()}")
df.head(3)


In [ ]:
# Verificamos la distribucion de categorias antes de modelar
print(df['categoria'].value_counts())
print(f"\nTotal de categorias: {df['categoria'].nunique()}")


In [ ]:
# Eliminamos filas con titulo o texto nulo (buena practica antes de construir el corpus)
filas_antes = len(df)
df = df.dropna(subset=['titulo', 'texto']).reset_index(drop=True)
print(f"Filas antes: {filas_antes} | Filas despues de quitar nulos: {len(df)}")


## 3. Construir el corpus (`titulo` + `texto`)

Combinamos titulo y texto en una sola columna de texto (`corpus`), que sera la entrada para la vectorizacion.
El titulo suele contener palabras muy representativas de la categoria, por lo que enriquece la representacion final.


In [ ]:
df['corpus'] = (df['titulo'].astype(str) + ' ' + df['texto'].astype(str)).str.lower()

df[['titulo', 'texto', 'corpus']].head(3)


## 4. Division train/test (estratificada)

Usamos `stratify=y` para asegurar que la proporcion de cada categoria se mantenga igual tanto en el set de
entrenamiento como en el de prueba -- esto es clave cuando hay mas de 2 clases, para que la evaluacion sea representativa.

In [ ]:
X = df['corpus']
y = df['categoria']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Entrenamiento: {X_train.shape[0]} ejemplos")
print(f"Prueba: {X_test.shape[0]} ejemplos")
print("\nDistribucion en entrenamiento:")
print(y_train.value_counts(normalize=True).round(3))


## 4.1 Balanceo de clases por oversampling (solo en train)

Las categorias no tienen exactamente el mismo numero de ejemplos (ej. `frontend` tiene mas articulos que `cloud`).
Para que el modelo no aprenda a favorecer las categorias mayoritarias, sobremuestreamos (con reemplazo) el set de
**entrenamiento** hasta que todas las categorias tengan la misma cantidad de ejemplos que la categoria mas grande.

**Importante:** esto se aplica *solo* sobre `X_train`/`y_train`, nunca sobre el set de prueba -- balancear el test
distorsionaria la evaluacion, ya que dejaria de reflejar la distribucion real de categorias.

In [ ]:
train_df = pd.DataFrame({"corpus": X_train.values, "categoria": y_train.values})
max_size = train_df["categoria"].value_counts().max()

balanced_parts = []
for categoria, grupo in train_df.groupby("categoria"):
    balanced_parts.append(grupo.sample(n=max_size, replace=True, random_state=42))

train_balanced = pd.concat(balanced_parts).sample(frac=1, random_state=42).reset_index(drop=True)

X_train = train_balanced["corpus"]
y_train = train_balanced["categoria"]

print("Distribucion en entrenamiento tras el balanceo:")
print(y_train.value_counts())


## 5. Vectorizacion TF-IDF con bigramas

Configuramos el vectorizador con:
- `ngram_range=(1, 2)`: incluye palabras sueltas (unigramas) **y** pares de palabras (bigramas, ej. "api rest",
  "machine learning"), lo que ayuda al modelo a capturar terminos tecnicos compuestos como una sola unidad de significado.
- `max_features=10000`: tamano de vocabulario. Se evaluaron varias configuraciones (5000 vs 10000 vs 20000
  features, min_df 2 vs 3, trigramas, n-gramas de caracteres, mas regularizacion) y esta combinacion (10000
  features, min_df=2) fue la que mejor resultado dio de forma consistente en ambos idiomas (ver seccion 11).
- `min_df=2`: ignora terminos que aparecen en menos de 2 documentos (reduce ruido de palabras muy raras).
- `stop_words`: segun el `IDIOMA` seleccionado en la celda de configuracion -- `STOP_WORDS_EN` o `STOP_WORDS_ES`,
  importadas desde `shared/stop_words.py` del repo (lista curada por el equipo, ~750/~950 palabras, validada
  con mejora consistente de accuracy y confianza respecto a la lista anterior).

**Importante:** el vectorizador se ajusta (`fit`) solo con los datos de entrenamiento ya balanceados, y luego se
aplica (`transform`) a los datos de prueba -- nunca al reves, para evitar fuga de informacion (data leakage).


In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    min_df=2,
    stop_words=stop_words
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Forma de la matriz TF-IDF (entrenamiento): {X_train_tfidf.shape}")
print(f"Forma de la matriz TF-IDF (prueba): {X_test_tfidf.shape}")


In [ ]:
# Vistazo a algunos terminos (unigramas y bigramas) aprendidos por el vectorizador
vocab = vectorizer.get_feature_names_out()
bigramas = [t for t in vocab if ' ' in t]

print(f"Total de terminos en el vocabulario: {len(vocab)}")
print(f"Total de bigramas incluidos: {len(bigramas)}")
print("\nEjemplos de bigramas detectados:")
print(bigramas[:15])


## 6. Entrenamiento del modelo (Regresion Logistica)

Usamos `class_weight='balanced'` para que el modelo compense automaticamente cualquier ligero desbalance entre
categorias, dando mas peso relativo a las clases con menos ejemplos.

In [ ]:
modelo = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=1.0,
    random_state=42
)

modelo.fit(X_train_tfidf, y_train)
print("Modelo entrenado correctamente.")
print(f"Categorias aprendidas: {modelo.classes_.tolist()}")


## 7. Evaluacion del modelo

### 7.1 Predicciones y Accuracy

In [ ]:
y_pred = modelo.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy en el set de prueba: {accuracy:.4f} ({accuracy*100:.2f}%)")


### 7.2 Reporte de clasificacion (precision, recall, F1-score por categoria)

In [ ]:
print(classification_report(y_test, y_pred))


### 7.3 Matriz de confusion

Muestra, para cada categoria real (filas), cuantos ejemplos fueron clasificados en cada categoria predicha (columnas).
La diagonal principal representa los aciertos; los valores fuera de la diagonal muestran en que categorias se confunde el modelo.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
cm = confusion_matrix(y_test, y_pred, labels=modelo.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=modelo.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=True)
plt.title('Matriz de confusion -- TF-IDF + Regresion Logistica')
plt.tight_layout()
plt.show()


### 7.4 Probabilidad de confianza en las predicciones

El campo `probabilidad` que exige el JSON de salida de la API corresponde a la probabilidad maxima que el modelo
asigna a la clase predicha (`predict_proba`). Ejemplos del set de prueba.

In [ ]:
probabilidades = modelo.predict_proba(X_test_tfidf)

# Tomamos 5 ejemplos al azar del set de prueba para inspeccionar
muestra_idx = np.random.RandomState(42).choice(len(X_test), size=5, replace=False)

for i in muestra_idx:
    texto_original = X_test.iloc[i][:80] + "..."
    categoria_real = y_test.iloc[i]
    categoria_predicha = y_pred[i]
    prob_max = probabilidades[i].max()

    print(f"Texto: {texto_original}")
    print(f"  Real: {categoria_real} | Predicho: {categoria_predicha} | Confianza: {prob_max:.2%}")
    print("-" * 90)


In [ ]:
# Distribucion general de la confianza del modelo en sus predicciones
confianza_max = probabilidades.max(axis=1)

plt.figure(figsize=(8, 5))
plt.hist(confianza_max, bins=20, color='#4C72B0', edgecolor='black')
plt.title('Distribucion de la confianza (probabilidad maxima) del modelo')
plt.xlabel('Probabilidad de la clase predicha')
plt.ylabel('Cantidad de ejemplos')
plt.tight_layout()
plt.show()

print(f"Confianza promedio: {confianza_max.mean():.2%}")
print(f"Confianza minima: {confianza_max.min():.2%}")
print(f"Confianza maxima: {confianza_max.max():.2%}")


**Como interpretar esto:** mientras mas a la derecha este concentrado el histograma (cerca del 100%), mas seguro
esta el modelo de sus propias predicciones. Si ves muchos casos con confianza baja (ej. cerca del 30-40% en un problema
de 7 clases, que equivaldria casi a "adivinar"), es senal de que el dataset tiene textos ambiguos entre categorias, y
la forma de mejorarlo es la que vimos antes: mas ejemplos por categoria, bigramas (ya incluidos aqui), y textos con
vocabulario bien distintivo por tema -- no se ajusta manualmente el numero de confianza.

## 8. Palabras clave por prediccion (`informacion_adicional`)

Para el campo `informacion_adicional` del JSON de salida, extraemos los terminos con mayor peso TF-IDF dentro del
texto especifico que se esta clasificando -- es decir, las palabras mas "distintivas" de ese contenido en particular.

In [ ]:
def extraer_palabras_clave(texto, vectorizer, top_n=5):
    """Devuelve las top_n palabras/bigramas con mayor peso TF-IDF dentro de un texto dado."""
    vector = vectorizer.transform([texto.lower()])
    vocab = np.array(vectorizer.get_feature_names_out())

    indices_ordenados = vector.toarray()[0].argsort()[::-1]
    top_indices = [i for i in indices_ordenados if vector[0, i] > 0][:top_n]

    return vocab[top_indices].tolist()


# Ejemplo con un texto del set de prueba
ejemplo_texto = X_test.iloc[0]
palabras_clave = extraer_palabras_clave(ejemplo_texto, vectorizer, top_n=5)

print(f"Texto: {ejemplo_texto[:100]}...")
print(f"Palabras clave detectadas: {palabras_clave}")


## 9. Funcion de prediccion completa (simula la respuesta de la API)

Esta funcion arma exactamente la estructura JSON que espera el endpoint `POST /v1/contenido` de la API,
integrando categoria, probabilidad y palabras clave en un solo resultado.

In [ ]:
def predecir_contenido(titulo, texto, modelo, vectorizer, top_n_keywords=5):
    corpus = (titulo + " " + texto).lower()
    vector = vectorizer.transform([corpus])

    categoria = modelo.predict(vector)[0]
    probabilidad = modelo.predict_proba(vector).max()
    palabras_clave = extraer_palabras_clave(corpus, vectorizer, top_n=top_n_keywords)

    return {
        "categoria": categoria,
        "probabilidad": round(float(probabilidad), 4),
        "informacion_adicional": palabras_clave
    }


# Prueba con un ejemplo nuevo, similar al formato del README de la API
resultado = predecir_contenido(
    titulo="Introduccion a Spring Boot",
    texto="En este contenido se presentan los conceptos basicos para la creacion de APIs REST utilizando Java y Spring Boot.",
    modelo=modelo,
    vectorizer=vectorizer
)

print(json.dumps(resultado, indent=2, ensure_ascii=False))


## 10. Serializacion del modelo

Guardamos el modelo entrenado y el vectorizador TF-IDF por separado (ambos son necesarios para hacer predicciones),
junto con un archivo de mapeo de categorias, listos para que el equipo de Backend los cargue en la API.

In [ ]:
import os
model_dir = f"../model/{IDIOMA}"
os.makedirs(model_dir, exist_ok=True)

joblib.dump(modelo, f"{model_dir}/model.joblib")
joblib.dump(vectorizer, f"{model_dir}/vectorizer.joblib")

label_mapping = {i: categoria for i, categoria in enumerate(modelo.classes_)}
with open(f"{model_dir}/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2, ensure_ascii=False)

print(f"Modelo, vectorizador y mapeo de categorias guardados en {model_dir}/")
print(f"\nlabel_mapping.json: {label_mapping}")


In [ ]:
# Verificacion rapida: recargar los artefactos desde disco y confirmar que funcionan igual
modelo_cargado = joblib.load(f"{model_dir}/model.joblib")
vectorizer_cargado = joblib.load(f"{model_dir}/vectorizer.joblib")

resultado_verificacion = predecir_contenido(
    titulo="Automatizacion con Docker y CI/CD",
    texto="Este articulo explica como construir pipelines de integracion continua y despliegue continuo usando contenedores.",
    modelo=modelo_cargado,
    vectorizer=vectorizer_cargado
)

print(json.dumps(resultado_verificacion, indent=2, ensure_ascii=False))


## 11. Resumen

- Se construyo el corpus combinando `titulo` + `texto` sobre el dataset final de contenidos tecnicos con etiquetas
  puras (sin ambiguedad entre categorias), para el idioma seleccionado (`IDIOMA`).
- Se balanceo el set de entrenamiento por oversampling, para que ninguna categoria domine el aprendizaje.
- Se vectorizo con **TF-IDF (unigramas + bigramas, 10000 features, min_df=2)**, con stopwords en ingles o
  espanol segun corresponda.
- Se entreno un modelo de **Regresion Logistica** con balanceo de clases (`class_weight='balanced'`).
- Se evaluo con **accuracy, precision, recall, F1-score y matriz de confusion**.

### Historial de evaluacion de modelos (Sprint DS-01/DS-04)

Antes de llegar a esta configuracion final se evaluaron 6 alternativas contra el baseline original
(TF-IDF 5000/min_df=3 + LogReg: 76.24% EN / 74.71% ES), todas con la misma metodologia (split 80/20
estratificado + oversampling solo en train):

| Modelo | Accuracy EN | Accuracy ES |
|---|---|---|
| LinearSVC | 74.06% | 72.27% |
| XGBoost | 74.80% | 73.20% |
| Embeddings multilingues (MiniLM) + LogReg | 68.65% | 67.27% |
| ComplementNB | 75.56% | 74.17% |
| LinearSVC calibrado | 76.31% | 75.09% |
| Ensamble (LogReg + LinearSVC + ComplementNB) | 77.29% | 75.89% |
| **TF-IDF Config B (10000/min_df=2) + LogReg -- elegido** | **77.21%** | **75.78%** |

Ninguna alternativa supero de forma significativa a la Config B (el ensamble da +0.08/+0.11 pts, dentro del
margen de ruido para 6,318 ejemplos de test), y todas agregan mas complejidad, tiempo de entrenamiento o
dependencias para produccion. Por eso se elige mantener un solo modelo: **Logistic Regression sobre TF-IDF
ajustado (10000 features, min_df=2)**.

- Se genero una funcion de prediccion completa que simula la respuesta JSON de la API (`categoria`, `probabilidad`, `informacion_adicional`).
- El modelo, el vectorizador y el mapeo de categorias quedaron serializados en `model/<idioma>/`, listos para que el
  equipo de Backend los integre en la API.

**Para el otro idioma:** cambia `IDIOMA` en la celda de configuracion (seccion 1.1) y ejecuta todo el notebook de
nuevo (`Run All`) -- vas a obtener un segundo modelo, independiente, en `model/<el_otro_idioma>/`.
